# Lab 10 — Soft and Hard Clustering, Built from Scratch

**Class 10.** Guide: `practice/10_clustering.md`. Lecture: `slides/10_clustering.pdf`.
Demos: `01_hard_soft.ipynb` to `07_usecases.ipynb`. In class (60 minutes): Parts A to D; at home: Parts E and F.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready:
it draws the plots and prints the comparisons once your functions work.

**The rule of this lab.** Every clustering method answers with a label (hard) or a row of memberships (soft).
You write the core loop of four methods with NumPy and compare each with its library twin on the **same data**.
The notebook prints the difference; explain it (the start, the scale, the shape).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from kneeliverse import kneedle
from lab10_utils import (
    benchmark,
    entropy,
    news_matrix,
    plot_cost,
    plot_kdistance,
    plot_labels,
    ten_points,
    wine_matrix,
)
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
BLUE, ORANGE, RED = "#0072B2", "#E69F00", "#D55E00"

# Part A — From NMF to Soft and Hard Assignments (12 min)

NMF factors the tf-idf matrix of 1 000 news articles, $X^\top \approx WH$ (words $\times$ articles). Column $i$ of $H$
is article $i$ over the topics: normalised, it is a membership row $u_i$.

## A1. Memberships from $W$ and $H$

Implement `nmf_memberships(W, H)` (`W` is `(d, k)`, `H` is `(k, n)`; return `U` `(n, k)`, every row summing to 1).
The scale of a part is free, $(WD)(D^{-1}H) = WH$: first multiply the row $j$ of $H$ by the column sum of $W_{\cdot j}$,
then normalise each article.

In [ ]:
def nmf_memberships(W, H):
    raise NotImplementedError

## A2. Entropy and the hard label

Implement `entropy_bits(U)`: the Shannon entropy in bits of every row, $-\sum_k u_{ik}\log_2 u_{ik}$ (use
`np.clip(U, 1e-12, 1)` to avoid $\log 0$). The driver factors the matrix, compares the raw and the rescaled
$\arg\max$ with the true categories and lists the most ambiguous articles.

In [ ]:
def entropy_bits(U):
    raise NotImplementedError

In [ ]:
X, cat, words = news_matrix()
_, W, H, _ = pynnmf.nmf_hals(X.T, 5, n=200, seed=0, init="nndsvd")
U = nmf_memberships(W, H)
raw = H.T / H.T.sum(axis=1, keepdims=True)
ent = entropy_bits(U)
rows = [
    ("raw H, argmax", ari(cat, raw.argmax(1)), float(raw.max(1).mean())),
    ("rescaled, argmax", ari(cat, U.argmax(1)), float(U.max(1).mean())),
    ("k-means on tf-idf (hard)", ari(cat, KMeans(5, n_init=10, random_state=0).fit_predict(X)), 1.0),
]
print(pl.DataFrame(rows, schema=["memberships", "ARI to the categories", "mean top u"], orient="row"))
print(
    f"rows sum to 1: {np.allclose(U.sum(1), 1)}; articles with top membership below 0.5: {(U.max(1) < 0.5).sum()} of {len(U)}"
)
for j in range(5):
    print(f"topic {j}:", ", ".join(words[np.argsort(-W[:, j])[:6]]))
print(
    "most ambiguous articles (index, category, entropy):",
    [(int(i), cat[i], round(float(ent[i]), 2)) for i in np.argsort(-ent)[:3]],
)

**Questions A.**

1. Why does the raw $\arg\max$ of $H$ score so much worse than the rescaled one?
2. A hard clustering would give an ambiguous article one topic. What does the soft row add, and what does the
   $\arg\max$ throw away?
3. The rows of $U$ sum to 1 but they are not probabilities of a model. What are they?


# Part B — $k$-Means and $k$-Means++ by Hand (15 min)

$k$-means alternates two steps: assign each point to the nearest centroid, move each centroid to the mean of its points.
The start decides which local optimum is reached.

## B1. The start and the loop

Implement `kmeanspp_init(X, k, rng)` (the first centroid is a random point; each next one is drawn with probability
proportional to the squared distance to the nearest chosen centroid; return `(k, d)`) and
`kmeans_by_hand(X, k, init, seed=0, iters=100)` (`init` is `"random"` or `"k-means++"`; return the labels, the centroids and
the list of costs $J$ at every iteration; stop when the labels do not change). Use `np.random.default_rng(seed)`.

In [ ]:
def kmeanspp_init(X, k, rng):
    raise NotImplementedError

In [ ]:
def kmeans_by_hand(X, k, init="k-means++", seed=0, iters=100):
    raise NotImplementedError

The driver runs your loop on the ten points of the lecture and on the `varied` set (50 starts of each kind), and
compares the best cost with `KMeans`.

In [ ]:
P = ten_points()
lab, mu, costs = kmeans_by_hand(P, 2, init="random", seed=0)
print(f"ten points: J = {[round(c, 2) for c in costs]}, centroids {mu.round(2).tolist()}")
Xv, yv = benchmark("varied")
res = {}
for init in ("random", "k-means++"):
    runs = [kmeans_by_hand(Xv, 3, init=init, seed=s) for s in range(50)]
    res[init] = np.array([r[2][-1] for r in runs])
twin = KMeans(3, n_init=10, random_state=0).fit(Xv).inertia_
rows = [(i, v.min(), np.median(v), v.max(), float((v > v.min() * 1.05).mean())) for i, v in res.items()]
print(pl.DataFrame(rows, schema=["start", "best J", "median J", "worst J", "share > 5% above best"], orient="row"))
print(
    f"KMeans (n_init=10) inertia: {twin:.3f}; ARI of your best run to the truth: "
    f"{ari(yv, kmeans_by_hand(Xv, 3, seed=int(np.argmin(res['k-means++'])))[0]):.3f}"
)
f, ax = plt.subplots(1, 2, figsize=(5.4, 2.1))
plot_cost(costs, ax[0], ylabel="$J$")
plot_labels(P, lab, ax[1], "ten points")
plt.show()

**Questions B1.**

1. Why can $J$ never rise from one iteration to the next? Say what each of the two steps does to $J$.
2. How often does a random start end more than 5 % above the best cost, and what does $k$-means++ change?
3. Your best cost and `KMeans(n_init=10)` agree. Why is one start not enough in general?


# Part C — $k$ Against $\epsilon$: DBSCAN on the Moons (15 min)

$k$-means needs $k$ and cuts convex cells; DBSCAN needs a radius $\epsilon$ and $\mathrm{minPts}$ and follows density.

## C1. The $k$-distance graph and its knee

Implement `k_distance(X, m)` (the sorted distance of every point to its $m$-th nearest neighbour, the point itself
counting as the first; return a vector of length `n`) and `eps_from_knee(d)` (the value of the knee of the sorted
curve, found with `kneedle.knee(np.c_[np.arange(len(d)), d])`, which returns the index).

In [ ]:
def k_distance(X, m):
    raise NotImplementedError

In [ ]:
def eps_from_knee(d):
    raise NotImplementedError

## C2. DBSCAN by hand

Implement `dbscan_by_hand(X, eps, min_pts)` (return labels, `-1` for noise). A point is **core** if at least `min_pts`
points (itself included) lie within `eps`. Visit the points in index order; from an unvisited core point start a
cluster and grow it: add every point within `eps` of a core point of the cluster, and keep expanding from the new
core points. A border point joins the first cluster that reaches it.

In [ ]:
def dbscan_by_hand(X, eps, min_pts):
    raise NotImplementedError

The driver tunes $\epsilon$ on the moons and on the rings with your knee, compares your labels with `DBSCAN` and
shows what $k$-means does on the same sets.

In [ ]:
rows = []
f, ax = plt.subplots(2, 4, figsize=(8.6, 4.2))
for r, name in enumerate(("moons", "circles")):
    X, y = benchmark(name)
    d = k_distance(X, 5)
    eps = eps_from_knee(d)
    mine = dbscan_by_hand(X, eps, 5)
    twin = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    km = KMeans(2, n_init=10, random_state=0).fit_predict(X)
    wide = [ari(y, DBSCAN(eps=eps * f_, min_samples=5).fit_predict(X)) for f_ in (1.5, 2.5)]
    rows.append(
        (
            name,
            eps,
            len(set(mine)) - (-1 in mine),
            int((mine == -1).sum()),
            ari(y, mine),
            ari(mine, twin),
            ari(y, km),
            *wide,
        )
    )
    plot_kdistance(d, eps, ax[r, 0])
    plot_labels(X, km, ax[r, 1], f"{name}: k-means")
    plot_labels(X, mine, ax[r, 2], "your DBSCAN")
    plot_labels(X, DBSCAN(eps=eps * 1.5, min_samples=5).fit_predict(X), ax[r, 3], "eps x 1.5")
plt.tight_layout()
plt.show()
print(
    pl.DataFrame(
        rows,
        schema=[
            "set",
            "eps (knee)",
            "clusters",
            "noise",
            "ARI yours",
            "ARI vs DBSCAN",
            "ARI k-means",
            "ARI eps x1.5",
            "ARI eps x2.5",
        ],
        orient="row",
    )
)

**Questions C.**

1. $k$-means has ARI near 0 on the rings. Which assumption of $k$-means do the rings break?
2. Your labels and `DBSCAN` should agree (ARI 1.0) except possibly on border points. Why can border points differ?
3. On the rings the knee gives 11 clusters, and $1.5\epsilon$ (last column of panels) gives 2. What does the table say about
   $2.5\epsilon$, and what does that tell you about the knee as a rule?


# Part D — Gaussian Mixtures by Hand: the EM Step (15 min)

EM alternates the **E-step** (the membership $u_{ik}$ of every point in every component) and the **M-step** (weights,
means and covariances re-estimated with the memberships as weights).

## D1. One EM iteration

Implement `em_step(X, pi, mu, Sigma)` (`pi` `(k,)`, `mu` `(k, d)`, `Sigma` `(k, d, d)`; return the new `pi`, `mu`, `Sigma`,
the memberships `U` `(n, k)` and the log-likelihood of the parameters you were given). Use
`scipy.stats.multivariate_normal.pdf` for the densities; add `1e-6 * I` to every new covariance.
Then implement `n_params(k, d)`, the number of free parameters of a full-covariance mixture, and
`bic(loglik, k, d, n)` $= P\ln n - 2\ln\hat{L}$.

In [ ]:
from scipy.stats import multivariate_normal as mvn


def em_step(X, pi, mu, Sigma):
    raise NotImplementedError

In [ ]:
def n_params(k, d):
    raise NotImplementedError

In [ ]:
def bic(loglik, k, d, n):
    raise NotImplementedError

The driver starts from the $k$-means centroids on the anisotropic set, runs 60 of your iterations, compares with
`GaussianMixture` and scores $k = 1..6$ by BIC.

In [ ]:
X, y = benchmark("aniso")
start = KMeans(3, n_init=1, random_state=0).fit(X).cluster_centers_
pi, mu, Sigma = np.full(3, 1 / 3), start, np.array([np.cov(X.T)] * 3)
ll = []
for _ in range(60):
    pi, mu, Sigma, U, loglik = em_step(X, pi, mu, Sigma)
    ll.append(loglik)
gm = GaussianMixture(3, means_init=start, tol=1e-8, max_iter=300, random_state=0).fit(X)
print(
    f"log-likelihood {ll[0]:.1f} -> {ll[-1]:.1f}; never falls: {all(b >= a - 1e-9 for a, b in zip(ll, ll[1:], strict=False))}"
)
print(
    f"ARI to the truth: yours {ari(y, U.argmax(1)):.3f}, GaussianMixture {ari(y, gm.predict(X)):.3f}; "
    f"max |u - u_sklearn| = {np.abs(U - gm.predict_proba(X)).max():.1e}"
)
print(f"mean entropy of the rows: {entropy(U).mean():.3f} bits")
rows = []
for k in range(1, 7):
    g = GaussianMixture(k, n_init=3, random_state=0).fit(X)
    rows.append((k, n_params(k, 2), bic(g.score(X) * len(X), k, 2, len(X)), g.bic(X)))
print(pl.DataFrame(rows, schema=["k", "P", "your BIC", "sklearn BIC"], orient="row"))
f, ax = plt.subplots(1, 2, figsize=(5.4, 2.1))
plot_cost(ll, ax[0], ylabel="log-likelihood")
ax[1].scatter(
    *X.T, c=np.clip(U @ np.array([[0.0, 0.447, 0.698], [0.902, 0.624, 0.0], [0.0, 0.62, 0.451]]), 0, 1), s=6, lw=0
)
ax[1].set_xticks([]), ax[1].set_yticks([])
plt.show()

**Questions D.**

1. The log-likelihood never falls. Which of the two steps raises the bound and which one makes it tight?
2. $k$-means has ARI about 0.5 on this set and your mixture 1.0. Which parameters of the mixture give it the stretch?
3. Where is the minimum of the BIC, and why does it not keep falling as $k$ grows?


# Part E — Choosing $k$ on the Wine Data, at Home

6 497 wines, 11 standardised measurements. The first 1 599 rows are the red wines: `is_red` is **only** used at the end
to check the answer, never to fit.

## E1. Three criteria for $k$

Implement `choose_k(X, ks, seed=0)` on a random sample of 2 000 wines: return a dict with the $k$ picked by the
**elbow** (Kneedle on the $k$-means inertia over `ks`), by the **silhouette** (largest mean over $k \geq 2$) and by the
**BIC** of a full-covariance Gaussian mixture (smallest).

In [ ]:
def choose_k(X, ks=range(1, 9), seed=0):
    raise NotImplementedError

In [ ]:
Xw, red, names = wine_matrix()
picks = choose_k(Xw)
print("k picked by each criterion:", picks)
k2 = KMeans(2, n_init=10, random_state=0).fit_predict(Xw)
print(f"k = 2 against red/white: ARI {ari(red, k2):.3f}")

**Questions E.**

1. Do the three criteria agree? Which one would you trust for a customer-segmentation task, and why?
2. The silhouette prefers $k = 2$; what do the two clusters turn out to be?


# Part F — Challenge: the Wine Case Study

Six methods, one data set, no label in the fit. `compare(X, red, k)` is the tournament: fit $k$-means, Ward, DBSCAN,
a Gaussian mixture, fuzzy $c$-means and NMF (on the positive and negative parts of the standardised wines) with
$k = 2$, and return a table with the ARI to `is_red`, the silhouette and the number of noise points.

## F1. The tournament

Implement `compare(X, red, k=2, seed=0)` using the rows of the driver below (`DBSCAN` with `min_samples=22` and
$\epsilon$ from the knee of your `k_distance`, on a sample of 3 000 wines; `fcm` from `lab10_utils`; NMF with
`pynnmf.nmf_hals` and your `nmf_memberships`). Then write three sentences: which method you would give to the
winemaker, which soft rows you would look at first, and which method fails and why.

In [ ]:
def compare(X, red, k=2, seed=0):
    raise NotImplementedError

In [ ]:
print(compare(Xw, red))


**Going further.** Hold out a third of the wines, fit the Gaussian mixture on the rest and measure how many of the
held-out wines have an entropy above 0.5 bit. Then repeat with 2 000 wines of one colour only: what does BIC pick?